### Операции над тензорами

Выполните небольшие упражнения, которые помогут закрепить ваши навыки манипуляций над тензорами. Вам нужно дописать код в местах с `...` и добиться того, чтобы все тесты проходили.

In [ ]:
import torch

---
#### Упражнение 1. Среднее значение по столбцам

In [ ]:
torch.manual_seed(42)
x = torch.randint(10, size=(2, 3)).float()
x

In [ ]:
mean_by_column = ...
assert torch.allclose(
    mean_by_column, _expected := torch.tensor([3.0, 6.5, 5.5])
), f"{mean_by_column} != {_expected}"

---
#### Упражнение 2. Softmax + Argmax: предсказание классов

Нейронная сеть для классификации возвращает *логиты* — ненормализованные числа для каждого класса. Чтобы получить вероятности, нужно применить `softmax`, а чтобы получить предсказанный класс — `argmax`.

Дана матрица логитов для батча из 4 примеров и 5 классов. Получите:
1. Матрицу вероятностей `probs` (каждая строка — распределение вероятностей, сумма = 1)
2. Вектор предсказанных классов `predicted_classes`

In [ ]:
torch.manual_seed(42)
logits = torch.randn(4, 5)
print("Логиты:")
print(logits)

In [ ]:
probs = ...
predicted_classes = ...

# вероятности должны давать в сумме 1 по строкам
assert torch.allclose(probs.sum(dim=1), torch.ones(4)), "Сумма вероятностей по строке должна быть 1!"
# все вероятности >= 0
assert (probs >= 0).all(), "Вероятности не могут быть отрицательными!"
# предсказанные классы
assert predicted_classes.shape == torch.Size([4]), f"Ожидалась форма (4,), получена {predicted_classes.shape}"
assert torch.equal(
    predicted_classes, _expected := torch.tensor([0, 2, 1, 0])
), f"{predicted_classes} != {_expected}"
print("Вероятности:\n", probs)
print("Предсказанные классы:", predicted_classes)

---
#### Упражнение 3. Точность классификации

Посчитайте долю правильных ответов (*accuracy*), сравнив предсказанные классы с истинными метками.

In [ ]:
predicted_classes = torch.tensor([0, 2, 1, 0])
true_labels = torch.tensor([0, 2, 1, 3])

accuracy = ...
assert accuracy == 0.75, f"Ожидалась точность 0.75, получена {accuracy}"
print(f"Accuracy: {accuracy:.0%}")

---
#### Упражнение 4. Взвешенное среднее

В тензоре `w` находятся веса для расчёта взвешенных средних значений тензора `x` по строкам:

$$\bar{x}_i = \sum_{j=1}^n x_{ij} w_{ij}$$

Найдите эти взвешенные средние.

**NB:** Обратите внимание, что веса $w_i$, дающие в сумме по столбцам единицу, мы получили применением функции `torch.softmax` (или метода `.softmax`) к исходным ненормализованным весам. Эта особая функция нам позже пригодится в задаче многоклассовой классификации, чтобы моделировать вероятностное распределение над возможными классами.

In [ ]:
torch.manual_seed(42)
x = torch.randint(10, size=(2, 3)).float()
w = torch.randint(10, size=(2, 3)).float().softmax(dim=1)
print(x)
print(w)

In [ ]:
w_avg = ...
assert torch.allclose(
    w_avg, _expected := torch.tensor([6.8940, 5.9690])
), f"{w_avg} != {_expected}"

---
#### Упражнение 5. Stack: собираем батч из списка примеров

Часто данные приходят как список отдельных тензоров (например, список векторов-признаков). Нужно собрать их в один тензор, добавив новую ось для батча.

Подсказка: `torch.stack` создаёт новую размерность

In [ ]:
torch.manual_seed(42)
samples = [torch.randn(5) for _ in range(4)]
print("Количество примеров:", len(samples))
print("Форма одного примера:", samples[0].shape)

In [ ]:
batch = ...
assert batch.shape == torch.Size([4, 5]), f"Ожидалась форма (4, 5), получена {batch.shape}"
assert torch.allclose(batch[0], samples[0]), "Первый пример не совпадает!"
print("Форма батча:", batch.shape)

---
#### Упражнение 6. Cat: склеивание последовательностей

В рекуррентных сетях (RNN) и трансформерах часто нужно склеивать тензоры вдоль оси последовательности. Дано два батча последовательностей — склейте их в одну длинную последовательность.

Подсказка: `torch.cat` склеивает тензоры вдоль существующей оси

In [ ]:
torch.manual_seed(42)
# батч из 2 последовательностей длины 3, каждый элемент — вектор размера 4
seq1 = torch.randn(2, 3, 4)
# батч из 2 последовательностей длины 5
seq2 = torch.randn(2, 5, 4)
print("Первая часть:", seq1.shape)
print("Вторая часть:", seq2.shape)

In [ ]:
full_seq = ...
assert full_seq.shape == torch.Size([2, 8, 4]), f"Ожидалась форма (2, 8, 4), получена {full_seq.shape}"
assert torch.allclose(full_seq[:, :3, :], seq1), "Первая часть не совпадает!"
assert torch.allclose(full_seq[:, 3:, :], seq2), "Вторая часть не совпадает!"
print("Склеенная последовательность:", full_seq.shape)

---
#### Упражнение 7. Flatten: подготовка изображений для полносвязной сети

При работе с изображениями данные обычно приходят в формате `(B, C, H, W)` — батч, каналы, высота, ширина. Чтобы подать такой тензор на вход полносвязному слою, нужно "развернуть" пространственные размерности в один вектор: `(B, C*H*W)`.

Дан тензор формы `(4, 3, 8, 8)` — батч из 4 RGB-изображений 8x8. Преобразуйте его в форму `(4, 192)`.

Подсказка: используйте `.flatten()` с параметром `start_dim`, или `.view()` / `.reshape()`

In [ ]:
torch.manual_seed(42)
images = torch.randn(4, 3, 8, 8)
print("Исходная форма:", images.shape)

In [ ]:
flat_images = ...
assert flat_images.shape == torch.Size([4, 192]), f"Ожидалась форма (4, 192), получена {flat_images.shape}"
# проверяем, что данные не перемешались: первые пиксели первого канала
assert torch.allclose(flat_images[0, :3], images[0, 0, 0, :3]), "Данные перемешались при reshape!"
print("Новая форма:", flat_images.shape)

---
#### Упражнение 8. Permute: перестановка осей

Часто изображения приходят в формате *channels-last* `(B, H, W, C)` (например, из `PIL` или `numpy`), а PyTorch ожидает *channels-first* `(B, C, H, W)`. Переставьте оси тензора.

Подсказка: используйте `.permute()` — он принимает нужный порядок осей

In [ ]:
torch.manual_seed(42)
# изображения в формате channels-last (как в numpy/PIL)
images_nhwc = torch.randn(4, 8, 8, 3)
print("Channels-last:", images_nhwc.shape)

In [ ]:
images_nchw = ...
assert images_nchw.shape == torch.Size([4, 3, 8, 8]), f"Ожидалась форма (4, 3, 8, 8), получена {images_nchw.shape}"
# проверяем, что пиксели на месте
assert torch.allclose(images_nchw[0, :, 0, 0], images_nhwc[0, 0, 0, :]), "Данные перемешались!"
print("Channels-first:", images_nchw.shape)

---
#### Упражнение 9. Транспонирование матриц батча

Дан тензор формы `(B, M, N)` — батч из `B` матриц размера `M x N`. Транспонируйте каждую матрицу в батче, чтобы получить `(B, N, M)`.

Подсказка: `.transpose()` принимает два индекса осей, которые нужно поменять местами

In [ ]:
torch.manual_seed(42)
batch_matrices = torch.randn(3, 4, 5)
print("Исходная форма:", batch_matrices.shape)

In [ ]:
transposed = ...
assert transposed.shape == torch.Size([3, 5, 4]), f"Ожидалась форма (3, 5, 4), получена {transposed.shape}"
assert torch.allclose(transposed[0, :, 0], batch_matrices[0, 0, :]), "Транспонирование неверное!"
print("Транспонированная форма:", transposed.shape)

---
#### Упражнение 10. Зануление чётных элементов

В тензоре `x` — 4 вектора размера 7. В каждом векторе сделайте чётные элементы нулевыми.

Способов много, попробуйте разные!

In [ ]:
x = torch.arange(28).view(4, 7) + 1
print(x)

In [ ]:
# способ 1: использование broadcasting и маски, которая закрывает чётные элементы
mask = torch.tensor([(i+1) % 2 for i in range(7)])
print(mask)
x = ...

assert torch.allclose(
    x, _expected := torch.tensor([
        [ 1,  0,  3,  0,  5,  0,  7],
        [ 8,  0, 10,  0, 12,  0, 14],
        [15,  0, 17,  0, 19,  0, 21],
        [22,  0, 24,  0, 26,  0, 28]
    ])), f"{x} != {_expected}"

In [ ]:
# способ 2: изменение на месте с помощью среза
x = torch.arange(28).view(4, 7) + 1
x[...] = 0  # вставляем нули в нужные места

assert torch.allclose(
    x, _expected := torch.tensor([
        [ 1,  0,  3,  0,  5,  0,  7],
        [ 8,  0, 10,  0, 12,  0, 14],
        [15,  0, 17,  0, 19,  0, 21],
        [22,  0, 24,  0, 26,  0, 28]
    ])), f"{x} != {_expected}"

---
#### Упражнение 11 (бонус). Матрица попарных расстояний

Даны две матрицы `x` и `y`, нужно получить матрицу `d`, где `d[i, j]` — евклидово расстояние между векторами `x[i]` (строка `i` матрицы `x`) и `y[j]` (строка `j` матрицы `y`).

Подсказка 1: воспользуйтесь broadcasting и добавлением размерностей в исходные тензоры. Для этого могут пригодиться методы `.view` и `.unsqueeze`

Подсказка 2: можно не считать евклидово расстояние вручную, есть функция `torch.linalg.norm` (или метод `.norm`), в которую можно подать векторы поэлементных разностей (их вы и получите с помощью broadcasting, если правильно измените размерности)

In [ ]:
torch.manual_seed(42)
x = torch.randint(10, size=(2, 3)).float()
y = torch.randint(10, size=(3, 3)).float()
print(x)
print(y)

In [ ]:
pdist = ...
assert torch.allclose(
    pdist,
    _expected := torch.tensor(
        [
            [7.0000, 2.4495, 6.1644],
            [6.7082, 2.4495, 6.0000]
        ]
    ),
), f"{pdist} != {_expected}"

---
#### Упражнение 12 (бонус). Агрегация по группам (graph pooling)

В графовых нейросетях (GNN) граф состоит из узлов, каждый из которых описывается вектором признаков. Часто нужно работать с *батчем графов* одновременно, при этом графы имеют разное количество узлов. Типичный подход — объединить все узлы всех графов в один тензор и хранить отдельный вектор `batch`, указывающий, к какому графу принадлежит каждый узел.

Задача: для каждого графа в батче посчитайте **средний вектор признаков** его узлов.

```
Пример:
  Граф 0: узлы 0, 1, 2    (batch = [0, 0, 0, ...])
  Граф 1: узлы 3, 4        (batch = [... 1, 1, ...])
  Граф 2: узлы 5, 6, 7, 8  (batch = [... 2, 2, 2, 2])
  
  Результат: 3 вектора — средние по каждому графу
```

Подсказка: для каждого графа можно создать булеву маску `batch == i` и использовать её для выборки нужных узлов, а затем посчитать среднее.

In [ ]:
torch.manual_seed(42)
# 9 узлов, каждый с вектором из 4 признаков
node_features = torch.randn(9, 4)
# принадлежность узлов к графам: 3 узла в графе 0, 2 в графе 1, 4 в графе 2
batch = torch.tensor([0, 0, 0, 1, 1, 2, 2, 2, 2])
num_graphs = 3

print("Признаки узлов (9 x 4):")
print(node_features)
print("\nПринадлежность к графам:", batch)

In [ ]:
graph_features = ...  # форма: (3, 4)

assert graph_features.shape == torch.Size([3, 4]), f"Ожидалась форма (3, 4), получена {graph_features.shape}"
# проверяем вручную для графа 0
expected_graph0 = node_features[:3].mean(dim=0)
assert torch.allclose(graph_features[0], expected_graph0, atol=1e-5), "Среднее для графа 0 неверно!"
# проверяем для графа 1
expected_graph1 = node_features[3:5].mean(dim=0)
assert torch.allclose(graph_features[1], expected_graph1, atol=1e-5), "Среднее для графа 1 неверно!"
# проверяем для графа 2
expected_graph2 = node_features[5:9].mean(dim=0)
assert torch.allclose(graph_features[2], expected_graph2, atol=1e-5), "Среднее для графа 2 неверно!"
print("Средние признаки по графам (3 x 4):")
print(graph_features)